In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

X_train, X_test, y_train, y_test = joblib.load("../data/splits.joblib")

In [2]:
results = []

def evaluate(name, model, Xtr, ytr):
    model.fit(Xtr, ytr)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "Method": name,
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "AUPRC": average_precision_score(y_test, proba),
    })

In [3]:
evaluate("No fix (baseline)", LogisticRegression(max_iter=1000), X_train, y_train)
evaluate("Class weights", LogisticRegression(max_iter=1000, class_weight="balanced"), X_train, y_train)

In [4]:
X_sm, y_sm = SMOTE(random_state=42).fit_resample(X_train, y_train)
print(y_sm.value_counts())
evaluate("SMOTE", LogisticRegression(max_iter=1000), X_sm, y_sm)

Class
0    227451
1    227451
Name: count, dtype: int64


In [5]:
X_us, y_us = RandomUnderSampler(random_state=42).fit_resample(X_train, y_train)
print(y_us.value_counts())
evaluate("Undersampling", LogisticRegression(max_iter=1000), X_us, y_us)

Class
0    394
1    394
Name: count, dtype: int64


In [6]:
table = pd.DataFrame(results).round(3)
print(table)

              Method  Precision  Recall     F1  AUPRC
0  No fix (baseline)      0.829   0.643  0.724  0.739
1      Class weights      0.061   0.918  0.114  0.717
2              SMOTE      0.059   0.918  0.111  0.725
3      Undersampling      0.038   0.918  0.074  0.693
